# Vegetation Biophysical Products: NDVI, LAI & FAPAR

This notebook demonstrates searching, loading, and visualizing CGLOPS vegetation biophysical products from the **Copernicus Data Space (CDSE)**.

Products are discovered via the OData API and streamed as Cloud-Optimised GeoTIFFs (COGs) from S3 object storage, clipped to a bounding box on the fly.

- **CLMS NDVI v3** — Normalised Difference Vegetation Index (300 m, 10-daily)
- **CLMS LAI v2** — Leaf Area Index (300 m, 10-daily)
- **CLMS FAPAR v2** — Fraction of Absorbed Photosynthetically Active Radiation (300 m, 10-daily)

In [ ]:
%matplotlib widget

import warnings
warnings.filterwarnings("ignore")

from rs_tools.config import BoundingBox
from rs_tools.datasets.catalog import get, list_datasets
from rs_tools.datasets.loader import load_dataset, items_to_dataarray
from rs_tools.visualization.timeseries import plot_timeseries_slider, plot_timeseries_line
from rs_tools.visualization.globe import add_globe_inset
from rs_tools.visualization.slider import slider_comparison
from rs_tools.visualization.rgb_composite import make_rgb, multi_temporal_rgb, plot_rgb

import matplotlib.pyplot as plt
import numpy as np

## 1. Browse the catalog

List all biophysical CLMS products registered in the catalog.

In [ ]:
for ds in list_datasets(tag="biophysical"):
    archives = ", ".join(ds.archive_collections.keys())
    res = ds.spatial_resolution or ""
    freq = ds.temporal_resolution or ""
    print(f"{ds.short_name:25s} {res:>8s}  {freq:>10s}  archives: {archives}")

## 2. Define area of interest

A bounding box over Belgium for this demonstration.

In [ ]:
bbox = BoundingBox(west=3.0, south=50.0, east=6.0, north=51.5)
START_DATE = "2023-01-01"
END_DATE = "2023-12-31"
print(f"AOI: {bbox}")
print(f"Period: {START_DATE} → {END_DATE}")

## 3. Load NDVI from CDSE

Load CLMS NDVI v3 (300 m, 10-daily) — the pipeline searches OData, acquires temporary S3 credentials, and streams COG tiles for the AOI.

In [ ]:
ndvi_info = get("CLMS_NDVI_V3")
print(f"Product : {ndvi_info.name}")
print(f"CDSE IDs: {ndvi_info.archive_collections.get('cdse', [])}")
print(f"Global:   {ndvi_info.is_global}")

ndvi_items = load_dataset(
    "CLMS_NDVI_V3",
    bbox=bbox,
    start_date=START_DATE,
    end_date=END_DATE,
    limit=40,
)
print(f"\nLoaded {len(ndvi_items)} NDVI dekads")
for item in ndvi_items[:5]:
    print(f"  {item.label}  shape={next(iter(item.data.values())).shape}")

## 4. Load LAI and FAPAR

Load complementary vegetation property products for the same region and period.

In [ ]:
products = {"LAI": "CLMS_LAI_V2", "FAPAR": "CLMS_FAPAR_V2"}
loaded = {}

for label, short_name in products.items():
    items = load_dataset(
        short_name,
        bbox=bbox,
        start_date=START_DATE,
        end_date=END_DATE,
        limit=40,
    )
    loaded[label] = items
    print(f"{label}: {len(items)} dekads")

## 5. Inspect loaded items

Examine the structure of loaded items — each has a datetime, platform label, and a dict of clipped DataArrays.

In [ ]:
# Show details for the first NDVI item
if ndvi_items:
    first = ndvi_items[0]
    print(f"ID:       {first.id}")
    print(f"Datetime: {first.datetime}")
    print(f"Platform: {first.platform}")
    print(f"CRS:      {first.crs}")
    print(f"Assets:   {list(first.data.keys())}")
    da = next(iter(first.data.values()))
    print(f"Shape:    {da.shape}  dtype={da.dtype}")

In [ ]:
# Show details for the first LAI item
if loaded.get("LAI"):
    first = loaded["LAI"][0]
    print(f"ID:       {first.id}")
    print(f"Datetime: {first.datetime}")
    da = next(iter(first.data.values()))
    print(f"Shape:    {da.shape}  dtype={da.dtype}")

## 6. Visualize temporal coverage

Plot a timeline showing when data was loaded for each product.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))

y_pos = 0
all_products = {"NDVI": ndvi_items, **{k: v for k, v in loaded.items()}}

for label, items in all_products.items():
    dates = [it.datetime for it in items if it.datetime]
    if dates:
        ax.scatter(dates, [y_pos] * len(dates), marker="|", s=200, label=f"CLMS {label}")
        y_pos += 1

ax.set_yticks(range(y_pos))
ax.set_yticklabels([])
ax.legend(loc="upper left")
ax.set_title("Temporal coverage — Belgium AOI")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 7. Location context — 3-D globe inset

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(bbox.west - 1, bbox.east + 1)
ax.set_ylim(bbox.south - 1, bbox.north + 1)
ax.set_title("Area of Interest — Belgium")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

from matplotlib.patches import Rectangle
rect = Rectangle(
    (bbox.west, bbox.south),
    bbox.east - bbox.west,
    bbox.north - bbox.south,
    linewidth=2, edgecolor="red", facecolor="red", alpha=0.2,
)
ax.add_patch(rect)
add_globe_inset(fig, bbox)
plt.show()

## Next steps

- **Time-series slider** (`plot_timeseries_slider`) for stepping through NDVI maps over time
- **Slider comparison** between NDVI and LAI or FAPAR for the same date
- **Multi-temporal RGB** composite from three NDVI dates
- **Explore other products**: GPP, NPP, ETA, SWI, Burnt Area — see `cglops_animations/` for GIF workflows